# Multiple Variables

### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
import climakitae as ck

from climakitae.core.data_load import load

import xarray as xr
import pandas as pd
import geopandas as gpd
from shapely.geometry import mapping
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from climakitae.new_core.data_access.boundaries import Boundaries
import intake

from climakitae.new_core.data_access.data_access import DataCatalog

### Step 1: Develop Approach

Check out https://github.com/cal-adapt/cae-notebooks/blob/main/basic_data_access.ipynb

In [2]:
# Initialize the interface
cd = ck.ClimateData(verbosity=-1)  # only give warnings and errors

In [7]:
cd.reset()
climate_data = (
    cd.catalog("cadcat")  # Catalog name
    .activity_id("WRF")  # Downscaling method
    .experiment_id("ssp370")  # SSP scenario
    .institution_id("UCLA")  # Institution name
    .table_id("mon")  # Data frequency
    .grid_label("d02")  # Grid resolution
    .variable_id(["t2",'prec'])  # Variable name
).get()

hello there!:['t2', 'prec']
list of strings?:True
variable_id_cleaned:['t2', 'prec']
variable_id_type:list
required_params:['catalog', 'variable_id', 'grid_label', 'table_id']
cadcat
['t2', 'prec']
d02
mon
[]
catalog_key: cadcat
valid_query: {'activity_id': 'WRF', 'institution_id': 'UCLA', 'experiment_id': ['ssp370'], 'table_id': 'mon', 'grid_label': 'd02', 'variable_id': ['t2', 'prec']}
search_query: {'activity_id': 'WRF', 'institution_id': 'UCLA', 'experiment_id': ['ssp370'], 'table_id': 'mon', 'grid_label': 'd02', 'variable_id': ['t2', 'prec']}
effective_key: cadcat
2026-10-07 15:22:40 - climakitae.new_core.processors.filter_unadjusted_models - WARNING - 

Your query selected models that do not have a-priori bias adjustment. 
These models have been removed from the returned query. 
To include them, please add the following processor to your query: 
ClimateData().processes('filter_unadjusted_models': 'no')

2026-10-07 15:22:40 - climakitae.new_core.processors.processor_utils - WARNIN

In [4]:
climate_data

<xarray.Dataset> Size: 4GB
Dimensions:            (sim: 5, time: 1032, y: 340, x: 270)
Coordinates:
  * sim                (sim) object 40B 'wrf_ucla_taiesm1_ssp370_r1i1p1f1' .....
    Lambert_Conformal  int32 4B 1
    lakemask           (y, x) float32 367kB dask.array<chunksize=(262, 208), meta=np.ndarray>
    landmask           (y, x) float32 367kB dask.array<chunksize=(262, 208), meta=np.ndarray>
    lat                (y, x) float32 367kB dask.array<chunksize=(262, 208), meta=np.ndarray>
    lon                (y, x) float32 367kB dask.array<chunksize=(262, 208), meta=np.ndarray>
  * time               (time) datetime64[ns] 8kB 2014-09-01 ... 2100-08-01
  * x                  (x) float64 2kB -4.728e+06 -4.719e+06 ... -2.307e+06
  * y                  (y) float64 3kB -3.341e+05 -3.251e+05 ... 2.717e+06
Data variables:
    prec               (sim, time, y, x) float32 2GB dask.array<chunksize=(1, 615, 262, 208), meta=np.ndarray>
    t2                 (sim, time, y, x) float32 2GB dask.array<chunksize=(1, 615, 262, 208), meta=np.ndarray>
Attributes: (12/174)
    ADAPT_DT_MAX:                     72.0
    ADAPT_DT_MIN:                     36.0
    ADAPT_DT_START:                   54.0
    AERCU_FCT:                        1.0
    AERCU_OPT:                        0
    AER_ANGEXP_OPT:                   1
    ...                               ...
    intake_esm_dataset_key:           WRF.UCLA.TaiESM1.ssp370.mon.d02
    resolution:                       9 km
    update_attributes:                Process 'update_attributes' applied to ...
    filter_unadjusted_models:         Process 'filter_unadjusted_models' appl...
    drop_leap_days:                   Leap days (February 29) have been remov...
    concat:                           Process 'concat' applied to the data. T...

## Processors

You can further customize your data retrieval using `processors`, which perform operations on the data before it is returned to you. The available processors are: 

- **`concat`** - Concatenate datasets along specified dimensions, default behavior is to concatenate on "time" by combining historical and SSP datasets.
- **`filter_unadjusted_models`** - Remove or include unadjusted models (default: "yes" to remove).
- **`update_attributes`** - Updates the attributes of your dataset based on the processors applied.
- `time_slice` - Applies a time slice to the requested dataset.
- `warming_level` - Applies a global warming level approach (as separate from the default time based approach). Please see our <a href="https://analytics.cal-adapt.org/scientific-guidance/general-climate-guidance/using-climate-projections.html#how-should-a-user-choose-between-global-warming-levels-and-a-time-based-approach-to-planning" target="_blank">guidance</a> on the use of global warming levels.
- `clip` - Applies a spatial clipping to the requested dataset. Many types of spatial clipping are supported including point based, bounding box, user provided shape files, and built-in boundaries including states, CA counties, CA watersheds, CA electric and utilities areas, CA demand forecast zones, CA electric balancing authority areas, and CA census tracts.
- `convert_units` - converts the units of your dataset.
- `metric_calc` - applies metric calculations to your dataset such as min, max, mean, median, percentiles, and 1-in-X calculations. 
- `bias_adjust_model_to_station` - For working with gridded data bias adjusted to historical HADISD weather station data.
- `export` - Exports your requested dataset to a range of file formats.

The first three processors (bolded) are run by default every time that you retrieve data. Examples of other available processors can be found in the `climakitae` library documentation, or in other example notebooks. <br><br>
It's important to note that processors are applied as a **dictionary**. This enables you to add more than one processor to your chain of operations. 

In [8]:
# concat
# Default behavior of concatenating by `time`
cd.reset()
concat_by_time = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .experiment_id(["historical", "ssp370"])  # Retrieve historical and future data
    .table_id("mon")
    .grid_label("d01")
    .variable_id(["t2","prec"])  # Precipitation
    .processes({"concat": "time"})  # Concatenate along time dimension
    .get()
)
concat_by_time

hello there!:['t2', 'prec']
list of strings?:True
variable_id_cleaned:['t2', 'prec']
variable_id_type:list
required_params:['catalog', 'variable_id', 'grid_label', 'table_id']
cadcat
['t2', 'prec']
d01
mon
[]
catalog_key: cadcat
valid_query: {'activity_id': 'WRF', 'institution_id': 'UCLA', 'experiment_id': ['historical', 'ssp370'], 'table_id': 'mon', 'grid_label': 'd01', 'variable_id': ['t2', 'prec']}
search_query: {'activity_id': 'WRF', 'institution_id': 'UCLA', 'experiment_id': ['historical', 'ssp370'], 'table_id': 'mon', 'grid_label': 'd01', 'variable_id': ['t2', 'prec']}
effective_key: cadcat
2026-10-07 15:22:57 - climakitae.new_core.processors.filter_unadjusted_models - WARNING - 

Your query selected models that do not have a-priori bias adjustment. 
These models have been removed from the returned query. 
To include them, please add the following processor to your query: 
ClimateData().processes('filter_unadjusted_models': 'no')



<xarray.Dataset> Size: 653MB
Dimensions:            (sim: 5, time: 1440, y: 104, x: 109)
Coordinates:
  * sim                (sim) object 40B 'wrf_ucla_miroc6_ssp370_r1i1p1f1' ......
    Lambert_Conformal  int32 4B 1
    lakemask           (y, x) float32 45kB dask.array<chunksize=(104, 109), meta=np.ndarray>
    landmask           (y, x) float32 45kB dask.array<chunksize=(104, 109), meta=np.ndarray>
    lat                (y, x) float32 45kB dask.array<chunksize=(104, 109), meta=np.ndarray>
    lon                (y, x) float32 45kB dask.array<chunksize=(104, 109), meta=np.ndarray>
  * time               (time) datetime64[ns] 12kB 1980-09-01 ... 2100-08-01
  * x                  (x) float64 872B -6.285e+06 -6.24e+06 ... -1.425e+06
  * y                  (y) float64 832B -1.126e+06 -1.081e+06 ... 3.509e+06
Data variables:
    prec               (sim, time, y, x) float32 326MB dask.array<chunksize=(1, 408, 104, 109), meta=np.ndarray>
    t2                 (sim, time, y, x) float32 326MB dask.array<chunksize=(1, 408, 104, 109), meta=np.ndarray>
Attributes: (12/182)
    ADAPT_DT_MAX:                     360.0
    ADAPT_DT_MIN:                     120.0
    ADAPT_DT_START:                   180.0
    AERCU_FCT:                        1.0
    AERCU_OPT:                        0
    AER_ANGEXP_OPT:                   1
    ...                               ...
    historical_prepended:             True
    resolution:                       45 km
    concat:                           Process 'concat' applied to the data. T...
    update_attributes:                Process 'update_attributes' applied to ...
    filter_unadjusted_models:         Process 'filter_unadjusted_models' appl...
    drop_leap_days:                   Leap days (February 29) have been remov...

### HDP

In [ ]:
cd.reset()
climate_data = (
    cd.catalog("hdp")  # Catalog name
    .activity_id("WRF")  # Downscaling method
    .experiment_id("ssp370")  # SSP scenario
    .institution_id("UCLA")  # Institution name
    .table_id("mon")  # Data frequency
    .grid_label("d02")  # Grid resolution
    .variable_id(["t2", "prec"])  # Variable name
).get()